# 04 — End-to-End FDP & Physics-Informed ML (PIML) Showcase

This notebook demonstrates the complete **Field Development Planning (FDP)** and **Physics-Informed Machine Learning (PIML)** workflow in `quick_pp`, moving beyond traditional single-point deterministic petrophysics to a fully probabilistic, physically constrained, and simulator-ready evaluation:

1. **Multi-Well QA/QC & Stratigraphic Normalization** (`apply_environmental_corrections`, `normalize_by_zone`)
2. **Probabilistic CPI & Volumetric Uncertainty Engine** (`run_cpi_uncertainty` / `run_log_uncertainty`, `calc_stochastic_reservoir_summary`, `mc_volumetric_uncertainty`)
3. **Multi-Well Free Water Level (FWL) & Transition Zone Optimization** (`estimate_mdt_fwl`, `optimize_fwl`, `ContactOptimizer`, `plot_fwl_reconciliation`)
4. **Special Core Analysis (SCAL), Relative Permeability & `SWOF` Handover** (`correct_core_data`, `CoreyRelPerm`, `corey_relperm`, `brooks_corey_relperm`, `let_relperm`, `fit_relperm`, `generate_swof_table`)
5. **Static-to-Dynamic Log Blocking & Upscaling** (`block_logs` / `upscale_logs`, `cardwell_parsons_bounds`, `pore_volume_upscale`, `expand_blocked_logs`, `plot_upscaled_logs`)
6. **Physics-Informed Machine Learning (PIML) & Locally Weighted Conformal Prediction** (`generate_fe_features`, `ConformalRegressor`, `ConformalMultiOutputRegressor`, `EnforceMineralSum`, `enforce_physical_constraints`, `physics_informed_loss`, `predict_with_uncertainty`)

In [ ]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
for _cand in (Path.cwd(), Path.cwd().parent):
    if (_cand / "quick_pp" / "__init__.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quick_pp import (
    # 1. Multi-Well QA/QC & Normalization
    apply_environmental_corrections,
    normalize_by_zone,
    # 2. Probabilistic CPI & Volumetric Uncertainty
    run_cpi_uncertainty,
    run_log_uncertainty,
    calc_stochastic_reservoir_summary,
    mc_volumetric_uncertainty,
    # 3. Multi-Well FWL & Transition Zone Optimization
    ContactOptimizer,
    estimate_mdt_fwl,
    optimize_fwl,
    predict_sw_shm,
    analyze_transition_zone,
    plot_fwl_reconciliation,
    # 4. SCAL, Relative Permeability & Eclipse/Intersect SWOF
    CoreyRelPerm,
    corey_relperm,
    brooks_corey_relperm,
    let_relperm,
    fit_relperm,
    generate_swof_table,
    correct_core_data,
    pc_lab_to_reservoir,
    # 5. Static-to-Dynamic Log Blocking & Upscaling
    block_logs,
    upscale_logs,
    cardwell_parsons_bounds,
    pore_volume_upscale,
    expand_blocked_logs,
    plot_upscaled_logs,
    # 6. Physics-Informed ML & Conformal Prediction
    ConformalRegressor,
    ConformalMultiOutputRegressor,
    EnforceMineralSum,
    enforce_physical_constraints,
    physics_informed_loss,
)
from quick_pp.machine_learning import (
    generate_fe_features,
    compute_physics_sample_weights,
    predict_with_uncertainty,
)

## 1. Load Bundled Well Data & Apply Multi-Well QA/QC + Stratigraphic Normalization

We load the bundled `HW-29` dataset (`mock_data.parquet`) and construct a 3-well multi-vintage field dataset (`HW-29`, `HW-30`, `HW-31`) with realistic borehole washouts and vintage calibration drift across two geological zones (`UPPER_SAND`, `LOWER_SAND`).

In [ ]:
# Load bundled well log & core dataset
data_path = Path("../quick_pp/machine_learning/mock_data/mock_data.parquet")
if not data_path.exists():
    data_path = Path("quick_pp/machine_learning/mock_data/mock_data.parquet")

base_df = pd.read_parquet(data_path).sort_values("DEPTH").reset_index(drop=True)
mid_depth = float(base_df["DEPTH"].median())
base_df["ZONES"] = np.where(base_df["DEPTH"] <= mid_depth, "UPPER_RESERVOIR", "LOWER_RESERVOIR")
base_df["CALI"] = 8.5 + np.where((base_df["DEPTH"] > mid_depth - 30) & (base_df["DEPTH"] < mid_depth + 20), 2.2, 0.1)
base_df["BS"] = 8.5
base_df["RM"] = base_df["RT"] * 0.78
base_df["RXO"] = base_df["RT"] * 0.55

# Create a 3-well multi-vintage dataset to demonstrate stratigraphic normalization & multi-well FWL
rng = np.random.default_rng(42)
w1 = base_df.copy()
w1["WELL_NAME"] = "HW-29"

w2 = base_df.copy()
w2["WELL_NAME"] = "HW-30"
w2["DEPTH"] = w2["DEPTH"] + 18.0
w2["GR"] = w2["GR"] * 1.15 + 12.0  # Vintage tool drift in GR
w2["NPHI"] = w2["NPHI"] + 0.025    # Neutron calibration offset

w3 = base_df.copy()
w3["WELL_NAME"] = "HW-31"
w3["DEPTH"] = w3["DEPTH"] - 14.0
w3["GR"] = w3["GR"] * 0.88 - 6.0
w3["NPHI"] = w3["NPHI"] - 0.018

field_df = pd.concat([w1, w2, w3], ignore_index=True)

# 1A. Apply automated borehole environmental & invasion corrections
field_df = apply_environmental_corrections(
    field_df,
    bitsize_default=8.5,
    mud_weight=9.8,
    salinity_ppm=25000.0,
    inplace=False,
    suffix="_CORR",
)

# 1B. Zone-by-zone stratigraphic quantile normalization anchored to key well 'HW-29'
field_df, norm_report = normalize_by_zone(
    field_df,
    key_well="HW-29",
    zone_col="ZONES",
    logs=["GR", "NPHI"],
    quantiles=(0.05, 0.95),
    method="quantile",
    return_report=True,
)

display(norm_report.head(6))

## 2. Probabilistic CPI Uncertainty Propagation (`run_cpi_uncertainty`)

Instead of relying on a single deterministic pass, `run_cpi_uncertainty` (alias for `run_log_uncertainty`) uses **Latin Hypercube Sampling (`scipy.stats.qmc.LatinHypercube`)** over uncertain petrophysical parameters ($a, m, n, R_w, \rho_{ma}, \rho_{sh}$) to compute `P10`, `P50`, and `P90` log curves (`PHIT_P10/P50/P90`, `SW_P10/P50/P90`, `VCLAY_P10/P50/P90`, `BVHC_P10/P50/P90`).

In [ ]:
# Run 150 Latin Hypercube realizations on key well HW-29
hw29_df = field_df[field_df["WELL_NAME"] == "HW-29"].copy()

param_bounds = {
    "a": (0.85, 1.15, 1.00, 0.06),
    "m": (1.80, 2.20, 1.95, 0.08),
    "n": (1.80, 2.20, 2.00, 0.08),
    "rw": (0.03, 0.12, 0.06, 0.02),
    "rho_ma": (2.64, 2.74, 2.68, 0.02),
    "rho_sh": (2.58, 2.76, 2.68, 0.04),
}

cpi_unc_df, realizations = run_cpi_uncertainty(
    hw29_df,
    model="ssc",
    saturation_method="waxman_smits",
    n_samples=150,
    param_bounds=param_bounds,
    seed=42,
    return_realizations=True,
)

# Inspect resulting P10 / P50 / P90 CPI columns
unc_cols = [
    "DEPTH", "PHIT_P10", "PHIT_P50", "PHIT_P90",
    "SW_P10", "SW_P50", "SW_P90",
    "BVHC_P10", "BVHC_P50", "BVHC_P90",
]
display(cpi_unc_df[unc_cols].head(8).round(4))

In [ ]:
# Plot P10-P50-P90 CPI Uncertainty Fan Chart for Porosity, Water Saturation, and Bulk Volume Hydrocarbon
fig, axes = plt.subplots(1, 3, figsize=(14, 6), sharey=True)
depth = cpi_unc_df["DEPTH"]

# Track 1: Total Porosity (PHIT)
axes[0].fill_betweenx(depth, cpi_unc_df["PHIT_P10"], cpi_unc_df["PHIT_P90"], color="royalblue", alpha=0.25, label="P10–P90 Band")
axes[0].plot(cpi_unc_df["PHIT_P50"], depth, color="navy", lw=1.4, label="PHIT P50")
if "CORE_POR" in cpi_unc_df.columns:
    axes[0].scatter(cpi_unc_df["CORE_POR"] / 100.0 if cpi_unc_df["CORE_POR"].max() > 1 else cpi_unc_df["CORE_POR"], depth, color="crimson", s=12, alpha=0.7, label="Core Porosity")
axes[0].set_xlabel("Total Porosity (v/v)")
axes[0].set_ylabel("Depth (ft)")
axes[0].set_title("PHIT Uncertainty (P10/P50/P90)")
axes[0].invert_yaxis()
axes[0].grid(True, ls="--", alpha=0.4)
axes[0].legend(loc="lower right", fontsize=8)

# Track 2: Water Saturation (SW)
axes[1].fill_betweenx(depth, cpi_unc_df["SW_P10"], cpi_unc_df["SW_P90"], color="teal", alpha=0.25, label="P10–P90 Band")
axes[1].plot(cpi_unc_df["SW_P50"], depth, color="darkslategray", lw=1.4, label="SW P50")
axes[1].set_xlabel("Water Saturation (v/v)")
axes[1].set_title("SW Uncertainty (P10/P50/P90)")
axes[1].grid(True, ls="--", alpha=0.4)
axes[1].legend(loc="lower left", fontsize=8)

# Track 3: Bulk Volume Hydrocarbon (BVHC)
axes[2].fill_betweenx(depth, cpi_unc_df["BVHC_P10"], cpi_unc_df["BVHC_P90"], color="seagreen", alpha=0.28, label="P10–P90 Band")
axes[2].plot(cpi_unc_df["BVHC_P50"], depth, color="darkgreen", lw=1.4, label="BVHC P50")
axes[2].set_xlabel("BVHC = PHIT * (1 - SW) (v/v)")
axes[2].set_title("Hydrocarbon Pore Volume (BVHC)")
axes[2].grid(True, ls="--", alpha=0.4)
axes[2].legend(loc="upper right", fontsize=8)

fig.suptitle("HW-29 Probabilistic CPI Evaluation (150 Latin Hypercube Realizations)", fontweight="bold")
fig.tight_layout()
fig.show()

In [ ]:
# Compute stochastic zone-level reservoir summary & volumetric STOIIP (P10 / P50 / P90)
stoch_res = calc_stochastic_reservoir_summary(
    hw29_df,
    cutoffs={"VSHALE": 0.40, "PHIT": 0.05, "SWT": 0.75},
    model="ssc",
    saturation_method="waxman_smits",
    n_samples=100,
    param_bounds=param_bounds,
    area_bound=(800.0, 2200.0, 1400.0, 300.0),
    volume_factor_bound=(1.15, 1.35),
    seed=42,
)

print("Zone-by-Zone Probabilistic STOIIP (MMbbl):")
for zone_name, vol_dict in stoch_res["volumetrics"].items():
    print(f"  {zone_name}: {vol_dict}")

display(stoch_res["summary_df"][stoch_res["summary_df"]["flag"] == "pay"])

## 3. Multi-Well Contact & Transition Zone Optimization (`optimize_fwl` & `ContactOptimizer`)

`optimize_fwl` and `ContactOptimizer` simultaneously solve for the field-wide **Free Water Level (FWL)**, **Oil-Water Contact (OWC)**, and **Top of Transition Zone (TTZ)** by reconciling:
- Resistivity-derived water saturation ($S_{w,\text{log}}$),
- Capillary pressure Saturation-Height Models ($S_{w,\text{SHM}}$ via Leverett-$J$, Skelt-Harrison, or Cuddy), and
- Wireline formation tester (`MDT`/`RFT`) pressure gradients (`estimate_mdt_fwl`).

In [ ]:
# Prepare 3-well dataset with realistic capillary transition zone & synthetic MDT pressure measurements
fwl_true = float(np.percentile(field_df["DEPTH"], 78))
field_df["PHIT"] = field_df["PHIE"].clip(0.04, 0.35)
field_df["PERM"] = field_df["CORE_PERM"].fillna(25.0).clip(0.5, 2000.0)
field_df["VSHALE"] = ((field_df["GR"] - 20.0) / 100.0).clip(0.0, 1.0)

# Generate synthetic MDT/RFT pressure survey across oil and water legs
mdt_depths = np.linspace(float(field_df["DEPTH"].min()) + 20.0, float(field_df["DEPTH"].max()) - 10.0, 14)
p_at_fwl = 3650.0
grad_oil = 0.345 * 0.433 / 0.433  # ~0.345 psi/ft
grad_wat = 1.05 * 0.433           # ~0.455 psi/ft
mdt_pressures = np.where(
    mdt_depths <= fwl_true,
    p_at_fwl + (mdt_depths - fwl_true) * grad_oil,
    p_at_fwl + (mdt_depths - fwl_true) * grad_wat,
)
pressure_df = pd.DataFrame({
    "TVDSS": mdt_depths,
    "FORMATION_PRESSURE": mdt_pressures + rng.normal(0, 0.35, size=len(mdt_depths)),
    "FLUID_TYPE": np.where(mdt_depths < fwl_true, "OIL", "WATER"),
})

mdt_info = estimate_mdt_fwl(pressure_df, unit="imperial")
print(f"MDT-derived FWL: {mdt_info['fwl']:.2f} ft (Pressure @ FWL: {mdt_info['pressure_at_fwl']:.1f} psi)")

# Simulate realistic resistivity Sw profile honoring Leverett-J around fwl_true with measurement noise
shm_true = {"a": 0.18, "b": 1.65, "swirr": 0.12}
field_df["SWT"] = np.clip(
    predict_sw_shm(
        field_df,
        fwl=fwl_true,
        model="leverett_j",
        shm_params=shm_true,
        depth_col="DEPTH",
        phit_col="PHIT",
        perm_col="PERM",
    ) + rng.normal(0.0, 0.035, size=len(field_df)),
    0.08,
    1.0,
)

# Run simultaneous multi-well FWL optimization reconciled with MDT pressure data
fwl_result = optimize_fwl(
    field_df,
    model="leverett_j",
    shm_params=shm_true,
    depth_col="DEPTH",
    sw_col="SWT",
    phit_col="PHIT",
    perm_col="PERM",
    well_col="WELL_NAME",
    vshale_col="VSHALE",
    pressure_df=pressure_df,
    mdt_penalty_weight=0.15,
    pore_volume_weighted=True,
    solve_per_well=True,
)

print(f"Optimized Multi-Well FWL : {fwl_result['fwl']:.2f} ft")
print(f"Oil-Water Contact (OWC)  : {fwl_result['owc']:.2f} ft (Capillary Fringe = {fwl_result['capillary_entry_height']:.2f} ft)")
print(f"Top Transition Zone (TTZ): {fwl_result['ttz']:.2f} ft (TZ Thickness = {fwl_result['transition_zone_thickness']:.2f} ft)")
print(f"FWL vs MDT Difference    : {fwl_result['fwl_reconciliation_diff']:+.2f} ft | Global Sw RMSE = {fwl_result['rmse']:.4f}, R² = {fwl_result['r2']:.3f}")

display(fwl_result["well_metrics"].round(3))

In [ ]:
# Plot 3-Panel Multi-Well Contact & Transition Zone Reconciliation Figure
fig_fwl = plot_fwl_reconciliation(
    fwl_result,
    depth_col="DEPTH",
    sw_col="SWT",
    well_col="WELL_NAME",
    title="COSTA Multi-Well FWL, OWC & Transition Zone Reconciliation",
)
plt.show()

## 4. Special Core Analysis (SCAL), `CoreyRelPerm` & Simulator `SWOF` Handover

`quick_pp.scal` bridges laboratory core measurements and dynamic reservoir simulation:
- **Core Plug Corrections** (`correct_core_data`): Overburden stress compaction, Klinkenberg gas slippage, and Dean-Stark clay-bound water (CBW) reconciliation.
- **Relative Permeability Modeling & Fitting** (`CoreyRelPerm`, `corey_relperm`, `brooks_corey_relperm`, `let_relperm`, `fit_relperm`).
- **Dynamic Simulator Handover** (`generate_swof_table` / `CoreyRelPerm.to_swof`): Generates monotonic `SWOF` tables (`SW`, `KRW`, `KROW`, `PCOW`) and Eclipse/Intersect keyword strings.

In [ ]:
# 4A. Correct ambient laboratory core plugs to reservoir net confining stress (NCS = 3200 psi) & Klinkenberg liquid perm
core_sub = hw29_df[hw29_df["CORE_POR"].notna() & hw29_df["CORE_PERM"].notna()].head(12).copy()
core_sub["CPORE"] = np.where(core_sub["CORE_POR"] > 1.0, core_sub["CORE_POR"] / 100.0, core_sub["CORE_POR"])
core_sub["CPERM"] = core_sub["CORE_PERM"].clip(lower=0.1)
core_sub["CSW"] = core_sub["SW"].clip(0.1, 0.95)
core_sub["PHIT"] = core_sub["CPORE"] + 0.02
core_sub["PHIE"] = core_sub["CPORE"]

core_corrected = correct_core_data(
    core_sub,
    net_confining_stress=3200.0,
    mean_pore_pressure=45.0,
    cbw=True,
    compaction_coeff=4.5e-6,
    perm_compaction_coeff=3.5e-5,
    compaction_method="exponential",
    klinkenberg_method="jones_owens",
    correct_dean_stark=True,
    poro_col="CPORE",
    perm_col="CPERM",
    sw_col="CSW",
)

display(core_corrected[["DEPTH", "CPORE_AMB", "CPORE", "CPERM_AIR", "CPERM", "CSW_RAW", "CSW"]].head(6).round(4))

In [ ]:
# 4B. Calibrate CoreyRelPerm & LET models to laboratory SCAL relative permeability measurements
sw_lab = np.linspace(0.16, 0.78, 12)
true_curves = corey_relperm(sw_lab, swirr=0.16, sor=0.22, krw_max=0.34, kro_max=0.88, nw=3.2, no=2.6)
krw_lab = np.clip(true_curves["krw"] + rng.normal(0, 0.008, len(sw_lab)), 0.0, 1.0)
kro_lab = np.clip(true_curves["kro"] + rng.normal(0, 0.012, len(sw_lab)), 0.0, 1.0)

# Fit using the object-oriented CoreyRelPerm estimator
corey_model = CoreyRelPerm(mu_w=0.45, mu_o=1.40).fit(sw_lab, krw=krw_lab, kro=kro_lab, swirr=0.16, sor=0.22)
print("Fitted CoreyRelPerm Parameters:", corey_model.fit_result_)

# Also compare with LET and Brooks-Corey models across a fine saturation grid
sw_grid = np.linspace(0.16, 0.78, 100)
corey_eval = corey_model(sw_grid)
let_fit = fit_relperm(sw_lab, krw=krw_lab, kro=kro_lab, model="let", swirr=0.16, sor=0.22)
let_eval = let_relperm(
    sw_grid, swirr=let_fit["swirr"], sor=let_fit["sor"],
    krw_max=let_fit["krw_max"], kro_max=let_fit["kro_max"],
    lw=let_fit["lw"], ew=let_fit["ew"], tw=let_fit["tw"],
    lo=let_fit["lo"], eo=let_fit["eo"], to=let_fit["to"],
)
bc_eval = brooks_corey_relperm(sw_grid, swirr=0.16, sor=0.22, krw_max=corey_model.krw_max, kro_max=corey_model.kro_max, lam=2.2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(sw_lab, krw_lab, color="royalblue", s=35, zorder=5, label="Lab krw")
axes[0].scatter(sw_lab, kro_lab, color="forestgreen", s=35, zorder=5, label="Lab kro")
axes[0].plot(sw_grid, corey_eval["krw"], "b-", lw=2, label=f"CoreyRelPerm krw (nw={corey_model.nw:.2f})")
axes[0].plot(sw_grid, corey_eval["kro"], "g-", lw=2, label=f"CoreyRelPerm kro (no={corey_model.no:.2f})")
axes[0].plot(sw_grid, let_eval["krw"], "b--", lw=1.4, alpha=0.8, label="LET Fit krw")
axes[0].plot(sw_grid, let_eval["kro"], "g--", lw=1.4, alpha=0.8, label="LET Fit kro")
axes[0].set_xlabel("Water Saturation Sw (v/v)")
axes[0].set_ylabel("Relative Permeability")
axes[0].set_title("SCAL Relative Permeability Calibration (Corey vs. LET)")
axes[0].grid(True, ls="--", alpha=0.4)
axes[0].legend(fontsize=8)

axes[1].plot(sw_grid, corey_eval["fw"], color="darkorange", lw=2.2, label="Corey Fractional Flow fw")
axes[1].plot(sw_grid, bc_eval["fw"], color="purple", ls="--", lw=1.6, label="Brooks-Corey fw (λ=2.2)")
axes[1].set_xlabel("Water Saturation Sw (v/v)")
axes[1].set_ylabel("Fractional Water Cut fw (v/v)")
axes[1].set_title("Fractional Flow Curve (fw vs. Sw)")
axes[1].grid(True, ls="--", alpha=0.4)
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

# Generate Eclipse / Intersect SWOF table
swof_out = corey_model.to_swof(n_points=12, pc_entry=1.8, pc_exponent=1.6, pc_max=45.0)
print("\nGenerated Eclipse SWOF Keyword Preview:")
print("\n".join(swof_out["swof_keyword"].splitlines()[:16]))

## 5. Static-to-Dynamic Log Blocking & Upscaling (`block_logs`)

`block_logs` (alias for `upscale_logs`) upscales fine-scale CPI logs onto cellular geomodel grids using petrophysically rigorous averaging:
- **Thickness-weighted arithmetic mean** for porosity (`PHIT`, `PHIE`) and shale volume (`VCLAY`),
- **Pore-volume weighted saturation mean** (`pore_volume_upscale`) for water saturation (`SWT`), strictly conserving hydrocarbon pore volume (`BVHC`),
- **Cardwell-Parsons (1945) directional permeability bounds** (`PERM_HARM` $k_v$, `PERM_GEOM` $k_{\text{eff}}$, `PERM_POWER` $k_\omega$, `PERM_ARITH` $k_h$, and `KV_KH_RATIO`), and
- **Majority vote** for discrete rock types (`ROCK_FLAG`).

In [ ]:
# Prepare fine-scale evaluated well log DataFrame for blocking
fine_df = cpi_unc_df.copy()
fine_df["PERM"] = fine_df["CORE_PERM"].fillna(25.0).clip(lower=0.05)
fine_df["ROCK_FLAG"] = np.where(fine_df["PERM"] >= 100.0, 1, np.where(fine_df["PERM"] >= 15.0, 2, 3))

# Block fine-scale logs onto 15-ft cellular geomodel layers respecting zone boundaries
blocked_df = block_logs(
    fine_df,
    cell_size=15.0,
    by_zone=True,
    depth_col="DEPTH",
    zone_col="ZONES",
    vsh_col="VCLAY_P50",
    phi_col="PHIT_P50",
    phie_col="PHIE_P50",
    sw_col="SW_P50",
    perm_col="PERM",
    facies_cols=["ROCK_FLAG"],
    perm_method="power_law",
    omega=0.33,
    cutoffs={"VSHALE": 0.40, "PHIT": 0.05, "SWT": 0.80},
)

# Verify exact Bulk Volume Hydrocarbon (BVHC) conservation between fine-scale and blocked grid
fine_hcpv = float((fine_df["PHIT_P50"] * (1.0 - fine_df["SW_P50"]) * fine_df["DEPTH"].diff().bfill()).sum())
blocked_hcpv = float((blocked_df["BVHC"] * blocked_df["GROSS"]).sum())
print(f" Blocked {len(fine_df)} fine-scale samples into {len(blocked_df)} cellular grid blocks.")
print(f"   Fine-scale Cumulative HCPV : {fine_hcpv:.4f} ft | Blocked Grid Cumulative HCPV : {blocked_hcpv:.4f} ft")

display(blocked_df[[
    "CELL_ID", "TOP", "BOTTOM", "ZONES", "GROSS", "NET_PAY", "NTG_PAY",
    "PHIT_P50", "SW_P50", "BVHC", "PERM_HARM", "PERM_GEOM", "PERM_POWER", "PERM_ARITH", "KV_KH_RATIO", "ROCK_FLAG"
]].head(8).round(4))

In [ ]:
# Plot Fine-Scale CPI Logs vs. Blocked Cellular Geomodel Step Profiles
fig_blk = plot_upscaled_logs(
    fine_df,
    blocked_df,
    depth_col="DEPTH",
    vsh_col="VCLAY_P50",
    phi_col="PHIT_P50",
    sw_col="SW_P50",
    perm_col="PERM",
    title="HW-29 Fine-Scale CPI vs. 15-ft Blocked Geomodel Grid (Cardwell-Parsons Bounds)",
)
plt.show()

## 6. Physics-Informed Machine Learning (PIML) & Locally Weighted Conformal Prediction (`ConformalRegressor`)

`quick_pp.machine_learning` combines domain rock-physics feature engineering with **Locally Weighted Split & Cross-Conformal Prediction** and **Volumetric Closure Constraints**:
- **Physics Feature Engineering (`generate_fe_features`)**: Computes apparent matrix density (`RHOMA_APP`), acoustic impedance (`AI` with Gardner fallback), Burke's `M_LITH` & `N_LITH` slopes, `TIGHT_FLAG`, and `COAL_FLAG`.
- **Locally Weighted Conformal Prediction (`ConformalRegressor`, `ConformalMultiOutputRegressor`)**: Normalizes calibration residuals by the ensemble tree standard deviation $\hat{\sigma}(x)$ so heterogeneous or washout intervals automatically receive wider `P10`–`P90` bands than clean homogeneous sands, with finite-sample order-statistic quantile guards.
- **Physics Constraints (`EnforceMineralSum`, `enforce_physical_constraints`, `physics_informed_loss`)**: Enforces $\sum V_{\text{mineral}} + \text{PHIT} = 1.0$, $0 \le \text{PHIE} \le \text{PHIT}$, and clamps $S_w = 1.0$ in tight streaks.

In [ ]:
# 6A. Generate Physics-Informed Features on HW-29
fe_df = generate_fe_features(cpi_unc_df)
fe_df["PHIT"] = cpi_unc_df["PHIT_P50"]
fe_df["PHIE"] = cpi_unc_df["PHIE_P50"]
fe_df["SWT"] = cpi_unc_df["SW_P50"]
fe_df["VCLAY"] = cpi_unc_df["VCLAY_P50"]
fe_df["VSAND"] = np.clip((1.0 - fe_df["PHIT"] - fe_df["VCLAY"]) * 0.75, 0.0, 1.0)
fe_df["VSILT"] = np.clip(1.0 - fe_df["PHIT"] - fe_df["VCLAY"] - fe_df["VSAND"], 0.0, 1.0)

feature_cols = ["GR", "RHOB", "NPHI", "RT", "DPHI", "RHOMA_APP", "AI", "M_LITH", "N_LITH", "TIGHT_FLAG", "COAL_FLAG"]
target_cols = ["PHIT", "PHIE", "SWT", "VCLAY", "VSAND", "VSILT"]

clean_ml = fe_df.dropna(subset=feature_cols + target_cols).reset_index(drop=True)
X_all = clean_ml[feature_cols]
y_all = clean_ml[target_cols]

# Split into calibration/training and test sets
split_idx = int(len(clean_ml) * 0.70)
X_train, X_test = X_train_cal, X_test = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
y_train, y_test = y_all.iloc[:split_idx], y_all.iloc[split_idx:]
depth_test = clean_ml["DEPTH"].iloc[split_idx:].to_numpy()

# 6B. Single-target Locally Weighted ConformalRegressor on PHIT (alpha=0.20 -> P10-P90)
conf_phit = ConformalRegressor(alpha=0.20, random_state=42)
conf_phit.fit(X_train, y_train["PHIT"])
phit_pred, phit_p10, phit_p90 = conf_phit.predict_interval(X_test)

coverage_phit = float(np.mean((y_test["PHIT"] >= phit_p10) & (y_test["PHIT"] <= phit_p90)))
print(f"Single-Target ConformalRegressor (PHIT) | q_hat = {conf_phit.q_hat:.3f} | Empirical P10–P90 Coverage = {coverage_phit:.1%}")

# 6C. Multi-Target Physics-Constrained ConformalMultiOutputRegressor
piml_model = ConformalMultiOutputRegressor(
    alpha=0.20,
    cv=4,
    enforce_physics=True,
    target_names=target_cols,
    random_state=42,
)
piml_model.fit(X_train, y_train)
y_pred_arr, y_p10_arr, y_p90_arr = predict_with_uncertainty(piml_model, X_test, targets=target_cols, alpha=0.20)

pred_df = pd.DataFrame(y_pred_arr, columns=target_cols, index=X_test.index)
for i, col in enumerate(target_cols):
    pred_df[f"{col}_P10"] = y_p10_arr[:, i]
    pred_df[f"{col}_P90"] = y_p90_arr[:, i]

# Verify volumetric closure (VCLAY + VSAND + VSILT + PHIT == 1.0) and Physics-Informed Loss
closure_sum = pred_df["VCLAY"] + pred_df["VSAND"] + pred_df["VSILT"] + pred_df["PHIT"]
pi_loss = physics_informed_loss(y_test, pred_df[target_cols], target_names=target_cols, X=X_test)
print(f"PIML Multi-Output Closure Max Error: {np.max(np.abs(closure_sum - 1.0)):.2e} | Composite Physics-Informed Loss: {pi_loss:.5f}")
display(pred_df[["PHIT", "PHIT_P10", "PHIT_P90", "SWT", "SWT_P10", "SWT_P90", "VCLAY", "VSAND", "VSILT"]].head(6).round(4))

In [ ]:
# Plot Conformal P10-P90 Prediction Intervals & Volumetric Lithology Closure on Test Interval
fig, axes = plt.subplots(1, 3, figsize=(14, 5.5), sharey=True)

# Track 1: Conformal PHIT P10-P90 vs Ground Truth
axes[0].fill_betweenx(depth_test, pred_df["PHIT_P10"], pred_df["PHIT_P90"], color="dodgerblue", alpha=0.28, label="Conformal P10–P90")
axes[0].plot(pred_df["PHIT"], depth_test, color="navy", lw=1.6, label="PIML PHIT Pred")
axes[0].scatter(y_test["PHIT"], depth_test, color="crimson", s=12, alpha=0.75, label="Actual PHIT")
axes[0].invert_yaxis()
axes[0].set_xlabel("Total Porosity PHIT (v/v)")
axes[0].set_ylabel("Depth (ft)")
axes[0].set_title("Conformal PHIT (Locally Weighted)")
axes[0].grid(True, ls="--", alpha=0.4)
axes[0].legend(loc="lower right", fontsize=8)

# Track 2: Conformal SWT P10-P90 vs Ground Truth
axes[1].fill_betweenx(depth_test, pred_df["SWT_P10"], pred_df["SWT_P90"], color="mediumseagreen", alpha=0.28, label="Conformal P10–P90")
axes[1].plot(pred_df["SWT"], depth_test, color="darkgreen", lw=1.6, label="PIML SWT Pred")
axes[1].scatter(y_test["SWT"], depth_test, color="darkorange", s=12, alpha=0.75, label="Actual SWT")
axes[1].set_xlabel("Water Saturation SWT (v/v)")
axes[1].set_title("Conformal SWT (Tight-Streak Constrained)")
axes[1].grid(True, ls="--", alpha=0.4)
axes[1].legend(loc="lower left", fontsize=8)

# Track 3: Volumetric Closure Partition (VCLAY + VSILT + VSAND + PHIT = 1.0)
v1 = pred_df["VCLAY"].to_numpy()
v2 = v1 + pred_df["VSILT"].to_numpy()
v3 = v2 + pred_df["VSAND"].to_numpy()
v4 = v3 + pred_df["PHIT"].to_numpy()
axes[2].fill_betweenx(depth_test, 0, v1, color="#8c6d31", alpha=0.75, label="VCLAY")
axes[2].fill_betweenx(depth_test, v1, v2, color="#bd9e39", alpha=0.75, label="VSILT")
axes[2].fill_betweenx(depth_test, v2, v3, color="#e7ba52", alpha=0.80, label="VSAND")
axes[2].fill_betweenx(depth_test, v3, v4, color="#3182bd", alpha=0.75, label="PHIT (Pore Vol)")
axes[2].set_xlim(0.0, 1.0)
axes[2].set_xlabel("Cumulative Bulk Volume Fraction (v/v)")
axes[2].set_title("EnforceMineralSum Closure (∑V + PHIT = 1)")
axes[2].grid(True, ls="--", alpha=0.4)
axes[2].legend(loc="lower left", fontsize=8)

fig.suptitle("Physics-Informed ML (PIML) & Locally Weighted Conformal Prediction Results", fontweight="bold")
fig.tight_layout()
plt.show()